In [10]:
from dotenv import load_dotenv
from langchain_community.embeddings import DashScopeEmbeddings

import os
load_dotenv(override=True)

# 嵌入模型的初始化
embedding_model = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
)

/var/folders/cq/f0kgnnnd0fq9zflvq22wf1780000gn/T/ipykernel_22028/2662207442.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


In [1]:
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

In [2]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

default
mytest


In [3]:
db_name = "rag_demo"

if db_name not in existed_databases:
    client.create_database(db_name=db_name)

In [4]:
client.drop_database(db_name = db_name)

In [5]:
if db_name not in client.list_databases():
    client.create_database(db_name)
client.use_database(db_name=db_name)

In [6]:
collections = client.list_collections()

for coll in collections:
    print(coll)

In [7]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

In [8]:
client.drop_collection(collection_name=collection_name)

# 插入向量

In [9]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

In [11]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

vectors = embedding_model.embed_documents(texts)

data = [
    {
        "id" : i,
        "vector" : vectors[i],
        "text" : texts[i],
        "source" : "demo"
    } for i in range(len(texts))
]

insert_res = client.upsert(
    collection_name=collection_name,
    data=data,
)

print("insert result : ",insert_res)

insert result :  {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


In [12]:

iterator = client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)

        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row["text"]},source = {row["source"]}")

        i += 1

iterator.close()

第1条数据：
id : 0,vector = [-0.06995262950658798, -0.03449718654155731, -0.038969043642282486, 0.0352691151201725, -0.02789587341248989],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.0345064140856266, 0.05864095687866211, -0.012167001143097878, 0.03689992055296898, 0.0192620437592268],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.0401918925344944, -0.015303953550755978, 0.0043546287342906, -0.024416347965598106, 0.0164601169526577],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
第4条数据：
id : 3,vector = [-0.01210562139749527, 0.07837200909852982, -0.01580328308045864, 0.04114990681409836, 0.01612548530101776],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


In [13]:

res = client.get(
    collection_name=collection_name,
    ids=[0,1,2]
)

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])


3
第1条数据：
id : 0,vector = [-0.06995262950658798, -0.03449718654155731, -0.038969043642282486, 0.0352691151201725, -0.02789587341248989],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.0345064140856266, 0.05864095687866211, -0.012167001143097878, 0.03689992055296898, 0.0192620437592268],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.0401918925344944, -0.015303953550755978, 0.0043546287342906, -0.024416347965598106, 0.0164601169526577],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo


In [14]:
# 相似度检索
query = "什么是向量数据库？"
query_vector = embedding_model.embed_query(query)


results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["text","source","id"]
)

for res in results[0]:
    print(res)

{'id': 1, 'distance': 0.7234620451927185, 'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}}
{'id': 3, 'distance': 0.4631018042564392, 'entity': {'id': 3, 'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
{'id': 2, 'distance': 0.38832250237464905, 'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}
